# 🔮 AI-Driven Customer Churn Prediction System
### End-to-End ML Pipeline – Jupyter Notebook
---

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')
%matplotlib inline
print('Libraries loaded ✓')

## 1. Data Generation

In [ ]:
from data.generate_data import generate_dataset
df = generate_dataset()
print(df.shape)
df.head()

In [ ]:
print('Churn rate:', df['churn'].mean().round(4))
print('Missing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])

## 2. Preprocessing & Feature Engineering

In [ ]:
from utils.preprocessing import full_pipeline
df_proc, X_scaled, y, feat_cols, scaler, encoders = full_pipeline('../data/ecommerce_churn.csv')
print(f'Features: {len(feat_cols)}')
print(feat_cols)

## 3. Sentiment Analysis

In [ ]:
from utils.sentiment import add_sentiment_features
df_proc = add_sentiment_features(df_proc)
df_proc[['review_text','sentiment_label','polarity']].head(10)

## 4. Exploratory Data Analysis

In [ ]:
# Churn distribution
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
df_proc['churn'].value_counts().plot.pie(ax=ax[0], labels=['No Churn','Churn'],
    autopct='%1.1f%%', colors=['green','red'])
ax[0].set_title('Churn Distribution')
sns.histplot(data=df_proc, x='days_since_last_purchase', hue='churn', bins=40, ax=ax[1])
ax[1].set_title('Days Since Last Purchase vs Churn')
plt.tight_layout(); plt.show()

In [ ]:
# Correlation heatmap (top features)
num_df = df_proc.select_dtypes(include=np.number)
top_corr = num_df.corr()['churn'].abs().sort_values(ascending=False).head(12).index
fig, ax = plt.subplots(figsize=(10,8))
sns.heatmap(num_df[top_corr].corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
plt.title('Correlation Heatmap (Top Churn-Correlated Features)')
plt.tight_layout(); plt.show()

## 5. Model Training & Evaluation

In [ ]:
from models.train_models import train_all
trained, best_model, results, X_te, y_te = train_all(X_scaled, y, feat_cols)
res_df = pd.DataFrame(results)
res_df

In [ ]:
# Model comparison chart
res_df.set_index('model')[['accuracy','f1','roc_auc']].plot(
    kind='bar', figsize=(10,5), colormap='tab10')
plt.title('Model Comparison')
plt.xticks(rotation=30)
plt.tight_layout(); plt.show()

## 6. Customer Segmentation

In [ ]:
from models.segmentation import segment_customers
df_proc = segment_customers(df_proc)
print(df_proc['segment'].value_counts())
df_proc.groupby('segment')[['total_spending','purchase_frequency','churn']].mean().round(3)

## 7. Churn Prediction Example

In [ ]:
from models.segmentation import recommend

# High-risk customer example
sample = {
    'days_since_last_purchase': 400,
    'cart_abandonment_rate': 0.9,
    'support_interactions': 7,
    'avg_review_rating': 1.5,
    'purchase_frequency': 1,
    'total_spending': 100
}
result = recommend(sample, churn_prob=0.87)
print(f"Risk Level: {result['risk_level']}")
print(f"Churn Reasons: {result['churn_reasons']}")
print(f"Strategies: {result['retention_strategies']}")

## ✅ Pipeline Complete
Run `python train_pipeline.py` for the full automated pipeline,
then `streamlit run dashboard/app.py` for the interactive dashboard.